In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import itertools
from itertools import combinations

## 1.Discipline Attention

In [ ]:
def calculate_avg_discipline_attention(discipline_activities_dict):
    """
    calculate average values of discipline attention for individual papers
    
    parameters:
    discipline_activities_dict: disciplinary labor division
    
    return:
    float: discipline attention
    """
    if not discipline_activities_dict:
        return 0
    
    all_activities = set()
    for activity_list in discipline_activities_dict.values():
        all_activities.update(activity_list)
    
    K_dis = len(all_activities)
    N_dis = len(discipline_activities_dict)
    
    if K_dis == 0 or N_dis == 0:
        return 0
    
    total_attention = 0
    for activity_list in discipline_activities_dict.values():
        Ki_dis = len(set(activity_list))
        Gi_dis = Ki_dis / K_dis
        total_attention += Gi_dis
    
    avg_attention = total_attention / N_dis
    return round(avg_attention, 4)

## 2.Discipline Cooperativeness

In [ ]:
def get_discipline_similarity(a, b, similarity_matrix):
    """get similarity of each discipline pair"""
    try:
        if a in similarity_matrix.index and b in similarity_matrix.columns:
            return similarity_matrix.loc[a, b]
        elif b in similarity_matrix.index and a in similarity_matrix.columns:
            return similarity_matrix.loc[b, a]
        else:
            return 0
    except:
        return 0

def calculate_within_activity_cooperativeness_exact(discipline_activities_dict, similarity_matrix):
    """Within-activity Cooperativeness"""
   
    # activity-discipline mapping
    activity_disciplines = {}
    for discipline, activities in discipline_activities_dict.items():
        for activity in activities:
            if activity not in activity_disciplines:
                activity_disciplines[activity] = Counter()
            activity_disciplines[activity][discipline] += 1

    if not activity_disciplines:
        return 0

    weighted_similarity_sum = 0
    total_cooperativeness_freq = 0

 
    for activity, discipline_counter in activity_disciplines.items():
        disc_list = list(discipline_counter.keys())

        if len(disc_list) < 2:
            continue
        
        for i in range(len(disc_list)):
            for j in range(i+1, len(disc_list)):
                disc_a = disc_list[i]
                disc_b = disc_list[j]

                coop_freq = discipline_counter[disc_a] * discipline_counter[disc_b]

                s_ab = get_discipline_similarity(disc_a, disc_b, similarity_matrix)

                weighted_similarity_sum += s_ab * coop_freq
                total_cooperativeness_freq += coop_freq

    if total_cooperativeness_freq > 0:
        return weighted_similarity_sum / total_cooperativeness_freq
    else:
        return 0

def calculate_between_activity_cooperativeness_exact(discipline_activities_dict, similarity_matrix):
    """Between-activity Cooperativeness"""
    
    # activity sets of each discipline
    discipline_activities_sets = {}
    for discipline, activities in discipline_activities_dict.items():
        discipline_activities_sets[discipline] = set(activities)

    disciplines = list(discipline_activities_sets.keys())
    n_disciplines = len(disciplines)

    if n_disciplines < 2:
        return 0

    discipline_pairs = list(itertools.combinations(disciplines, 2))

    weighted_similarity_sum = 0
    total_cooperativeness_strength = 0

    for disc_a, disc_b in discipline_pairs:
        activities_a = discipline_activities_sets[disc_a]
        activities_b = discipline_activities_sets[disc_b]

        common_activities = activities_a & activities_b
        diff_activities_a = activities_a - common_activities
        diff_activities_b = activities_b - common_activities

        coop_strength = len(diff_activities_a) * len(diff_activities_b)

        if coop_strength > 0:
            s_ab = get_discipline_similarity(disc_a, disc_b, similarity_matrix)
            weighted_similarity_sum += s_ab * coop_strength
            total_cooperativeness_strength += coop_strength

    if total_cooperativeness_strength > 0:
        return weighted_similarity_sum / total_cooperativeness_strength
    else:
        return 0


def calculate_discipline_cooperativeness_exact(discipline_activities_dict, similarity_matrix):
    """
    calculate values of discipline cooperativeness for individual papers
    
    parameters:
    discipline_activities_dict: disciplinary labor division
    similarity_matrix: DataFrame，discipline similarity
    
    return:
    float: discipline cooperativeness
    """
    if not discipline_activities_dict or not isinstance(discipline_activities_dict, dict):
        return np.nan

    disciplines = list(discipline_activities_dict.keys())
    n_disciplines = len(disciplines)

    if n_disciplines < 2:
        return np.nan

    within_DC = calculate_within_activity_cooperativeness_exact(
        discipline_activities_dict, similarity_matrix
    )

    between_DC = calculate_between_activity_cooperativeness_exact(
        discipline_activities_dict, similarity_matrix
    )

    # weighted DC
    total_DC = 0.7 * within_DC + 0.3 * between_DC
    return round(total_DC, 4)

## 3.Discipline Concentration in Activities

In [ ]:
def calculate_discipline_concentration_activity(discipline_activities_dict):
    """
    calculate values of discipline concentration in activities for individual papers
    
    parameters:
    discipline_activities_dict: disciplinary labor division
    
    return:
    DCA: discipline concentration in activities
    """
    if not discipline_activities_dict:
        return np.nan
    

    activity_disciplines = {}    
    for discipline, activities in discipline_activities_dict.items():
        for activity in activities:
            if activity not in activity_disciplines:
                activity_disciplines[activity] = []
            activity_disciplines[activity].append(discipline)     
    
    if not activity_disciplines:
        return np.nan

    # total number of participations of each activity
    N_total = sum(len(disciplines) for disciplines in activity_disciplines.values())
    
    if N_total == 0:
        return np.nan
    
    DCA = 0
    
    # Simpson value of each activity
    for activity, disciplines in activity_disciplines.items():
        discipline_counts = Counter(disciplines)
        N_j = len(disciplines)
        simpson_index = 1 - sum((count/N_j)**2 for count in discipline_counts.values())
        # weighted Simpson value
        weight = N_j / N_total
        DCA += weight * simpson_index
    
    return DCA

## 4.Activity Cooperativeness

In [ ]:
def calculate_jaccard_similarity(set1, set2):
    """
    calculate Jaccard similarity
    
    parametera:
    set1, set2
    
    return:
    float: Jaccard similarity, [0,1]
    """
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    
    if union == 0:
        return 0.0
    else:
        return intersection / union

def calculate_activity_cooperativeness_jaccard(discipline_activities_dict):
    """
    calculate values of activity cooperativeness for individual papers
    
    parameters:
    discipline_activities_dict: disciplinary labor division
    
    return:
    float: activity cooperativeness
    """
    if not discipline_activities_dict:
        return 0
    
    activity_to_disciplines = {}
    for discipline, activities in discipline_activities_dict.items():
        for activity in set(activities):
            if activity not in activity_to_disciplines:
                activity_to_disciplines[activity] = set()
            activity_to_disciplines[activity].add(discipline)
    
    activities = list(activity_to_disciplines.keys())
    K = len(activities)
    
    if K <= 1:
        return 0 
    
    # calculate AC of each activity
    activity_cooperativeness_scores = []
    
    for i, activity_i in enumerate(activities):
        similarities = []
        for j, activity_j in enumerate(activities):
            if i != j:
                jaccard_sim = calculate_jaccard_similarity(
                    activity_to_disciplines[activity_i],
                    activity_to_disciplines[activity_j]
                )
                similarities.append(jaccard_sim)

        if similarities:
            Ij = np.mean(similarities)
            activity_cooperativeness_scores.append(Ij)
    
    # weighted AC
    if activity_cooperativeness_scores:
        Ia = np.mean(activity_cooperativeness_scores)
        return round(Ia, 4)
    else:
        return 0